# Stage 4: Full Experiment Matrix

This notebook runs experiments E1-E4 (chunking ablation) and E6-E7 (agent architectures) on all test claims and compares results systematically.

| Exp | Chunking | Retrieval | Agent | What It Tests |
|-----|----------|-----------|-------|--------------|
| E1 | fixed | naive | single_pass | Baseline |
| E2 | semantic | hybrid_reranked | single_pass | Best RAG |
| E3 | section_aware | hybrid_reranked | single_pass | Section chunking |
| E4 | recursive | hybrid_reranked | single_pass | Recursive + metadata |
| E6 | semantic | hybrid_reranked | langgraph_multi | Multi-agent |
| E7 | semantic | hybrid_reranked | strands_rerouting | Adaptive rerouting |

**Note:** E5 (Strands multi-agent) requires AWS Bedrock. E8-E12 require OpenAI/Ollama.

In [ ]:
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

with open("data/test_claims.json") as f:
    claims = json.load(f)

print(f"Running experiments on {len(claims)} claims")

## 4.1 Run all experiments

In [ ]:
from src.pipelines.configurable import run_experiment
from src.experiment_runner import EXPERIMENT_CONFIGS

# Select experiments that work with Anthropic API only
EXPERIMENTS_TO_RUN = ["E1", "E2", "E3", "E4", "E6", "E7"]

all_results = {}

for exp_id in EXPERIMENTS_TO_RUN:
    config = EXPERIMENT_CONFIGS[exp_id]
    pipeline_config = {k: v for k, v in config.items() if k != "name"}
    
    print(f"\n{'='*60}")
    print(f"{exp_id}: {config['name']}")
    print(f"  {pipeline_config}")
    print(f"{'='*60}")
    
    exp_results = []
    for c in claims:
        print(f"  {c['claim'][:45]}...", end=" ", flush=True)
        try:
            result = run_experiment(c["claim"], **pipeline_config)
            result["expected_verdict"] = c["expected_verdict"]
            result["correct"] = result["verdict"] == c["expected_verdict"]
            exp_results.append(result)
            status = "OK" if result["correct"] else "WRONG"
            print(f"→ {result['verdict']} ({status}) {result['metadata']['latency_seconds']}s")
        except Exception as e:
            print(f"→ ERROR: {e}")
            exp_results.append({"claim": c["claim"], "verdict": "ERROR", "explanation": str(e), "evidence": [],
                                "metadata": {"latency_seconds": 0}, "expected_verdict": c["expected_verdict"], "correct": False})
    
    correct = sum(1 for r in exp_results if r.get("correct"))
    total_time = sum(r["metadata"]["latency_seconds"] for r in exp_results)
    print(f"  Result: {correct}/{len(exp_results)} correct, {total_time:.1f}s total")
    
    all_results[exp_id] = exp_results

print("\nAll experiments complete!")

## 4.2 Accuracy comparison table

In [ ]:
accuracy_table = []
for exp_id in EXPERIMENTS_TO_RUN:
    results = all_results[exp_id]
    config = EXPERIMENT_CONFIGS[exp_id]
    correct = sum(1 for r in results if r.get("correct"))
    total_time = sum(r["metadata"]["latency_seconds"] for r in results)
    avg_time = total_time / len(results)
    
    accuracy_table.append({
        "Experiment": exp_id,
        "Name": config["name"],
        "Accuracy": f"{correct}/{len(results)}",
        "Accuracy %": round(correct / len(results) * 100, 1),
        "Avg Latency (s)": round(avg_time, 1),
        "Total Time (s)": round(total_time, 1),
    })

df = pd.DataFrame(accuracy_table)
df

## 4.3 Verdict matrix — which claims does each experiment get right?

In [ ]:
verdict_matrix = pd.DataFrame()
verdict_matrix["Claim"] = [c["claim"][:40] for c in claims]
verdict_matrix["Expected"] = [c["expected_verdict"] for c in claims]

for exp_id in EXPERIMENTS_TO_RUN:
    results = all_results[exp_id]
    verdict_matrix[exp_id] = [
        f"{r['verdict']} {'Y' if r.get('correct') else ''}"
        for r in results
    ]

verdict_matrix

## 4.4 Accuracy bar chart

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
accuracies = [sum(1 for r in all_results[e] if r.get("correct")) / len(all_results[e]) * 100 for e in EXPERIMENTS_TO_RUN]
ax1.bar(EXPERIMENTS_TO_RUN, accuracies, color=colors)
ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Verdict Accuracy by Experiment")
ax1.set_ylim(0, 100)
for i, v in enumerate(accuracies):
    ax1.text(i, v + 1, f"{v:.0f}%", ha="center", fontweight="bold")

# Latency
avg_latencies = [sum(r["metadata"]["latency_seconds"] for r in all_results[e]) / len(all_results[e]) for e in EXPERIMENTS_TO_RUN]
ax2.bar(EXPERIMENTS_TO_RUN, avg_latencies, color=colors)
ax2.set_ylabel("Avg Latency (seconds)")
ax2.set_title("Average Latency by Experiment")
for i, v in enumerate(avg_latencies):
    ax2.text(i, v + 0.5, f"{v:.1f}s", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 4.5 Accuracy vs Latency scatter (cost-quality frontier)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for i, (exp_id, color) in enumerate(zip(EXPERIMENTS_TO_RUN, colors)):
    acc = accuracies[i]
    lat = avg_latencies[i]
    ax.scatter(lat, acc, s=200, c=color, zorder=5)
    ax.annotate(f"{exp_id}\n{EXPERIMENT_CONFIGS[exp_id]['name'][:20]}",
                (lat, acc), textcoords="offset points", xytext=(10, 5), fontsize=9)

ax.set_xlabel("Average Latency (seconds)")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy vs Latency — Cost-Quality Frontier")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 4.6 Chunking ablation (E1 vs E2 vs E3 vs E4)

In [ ]:
print("Chunking Ablation (all single-pass, E1 uses naive, E2-E4 use hybrid_reranked):")
print("=" * 70)
chunking_exps = ["E1", "E2", "E3", "E4"]

for i, c in enumerate(claims):
    verdicts = [all_results[e][i]["verdict"] for e in chunking_exps]
    correct_flags = ["Y" if all_results[e][i].get("correct") else " " for e in chunking_exps]
    agreement = len(set(verdicts)) == 1
    print(f"{c['claim'][:40]:<42} Expected={c['expected_verdict']:<15}  "
          f"E1={verdicts[0]:<12}[{correct_flags[0]}]  E2={verdicts[1]:<12}[{correct_flags[1]}]  "
          f"E3={verdicts[2]:<12}[{correct_flags[2]}]  E4={verdicts[3]:<12}[{correct_flags[3]}]  "
          f"{'AGREE' if agreement else 'DIFFER'}")

print(f"\nAccuracies: E1={accuracies[0]:.0f}%  E2={accuracies[1]:.0f}%  E3={accuracies[2]:.0f}%  E4={accuracies[3]:.0f}%")

## 4.7 Agent architecture comparison (E2 vs E6 vs E7)

In [ ]:
print("Agent Architecture Comparison (all semantic + hybrid_reranked):")
print("=" * 70)
agent_exps = ["E2", "E6", "E7"]
agent_idx = [EXPERIMENTS_TO_RUN.index(e) for e in agent_exps]

for i, c in enumerate(claims):
    verdicts = [all_results[e][i]["verdict"] for e in agent_exps]
    latencies = [all_results[e][i]["metadata"]["latency_seconds"] for e in agent_exps]
    correct_flags = ["Y" if all_results[e][i].get("correct") else " " for e in agent_exps]
    
    print(f"{c['claim'][:40]:<42}  "
          f"SP={verdicts[0]:<15}[{correct_flags[0]}] {latencies[0]:>5.1f}s  "
          f"LG={verdicts[1]:<15}[{correct_flags[1]}] {latencies[1]:>5.1f}s  "
          f"RT={verdicts[2]:<15}[{correct_flags[2]}] {latencies[2]:>5.1f}s")

print(f"\nAccuracies: SP(E2)={accuracies[1]:.0f}%  LG(E6)={accuracies[4]:.0f}%  RT(E7)={accuracies[5]:.0f}%")
print(f"Avg latency: SP={avg_latencies[1]:.1f}s  LG={avg_latencies[4]:.1f}s  RT={avg_latencies[5]:.1f}s")

## 4.8 Save results for evaluation

In [ ]:
from pathlib import Path

output_dir = Path("results/experiments")
output_dir.mkdir(parents=True, exist_ok=True)

for exp_id in EXPERIMENTS_TO_RUN:
    results = all_results[exp_id]
    config = EXPERIMENT_CONFIGS[exp_id]
    correct = sum(1 for r in results if r.get("correct"))
    
    output = {
        "experiment_id": exp_id,
        "config": config,
        "total_claims": len(results),
        "correct": correct,
        "accuracy": correct / len(results),
        "results": results,
    }
    
    path = output_dir / f"{exp_id}.json"
    with open(path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"Saved {exp_id} to {path}")

print("\nDone! Results saved for Stage 5 evaluation.")

## Key Takeaways

**Chunking ablation (E1-E4):**
- Does semantic/section/recursive chunking improve over fixed?
- Is the improvement from better chunking or from better retrieval (hybrid vs naive)?

**Agent architecture (E2 vs E6 vs E7):**
- Does multi-agent improve verdict accuracy over single-pass?
- Does rerouting help on claims with weak initial evidence?
- Is the latency cost worth the quality gain?

**Next step:** Stage 5 runs formal evaluation metrics (Macro-F1, McNemar's test, LLM judge, bootstrap CIs).